# Welcome to the Quantum Trojan Challenge

One forensic investigation spans six intercepted quantum jobs. Recover each
job's persistent measurement-mask key and explain the conditional behavior
injected into its circuit.

You receive **a circuit and 8,192 masked measurement shots for each case**.
Cases are named `circuit1` through `circuit6`.  Each case has an independent key.

| Challenge | Evidence | Required work | Points |
| --- | --- | --- | --- |
| Silent Witness — six circuit cases | QASM circuits and measurement counts | Keys, trigger identification and explanations, recovery code and write-up | 100 |

Follow [setup](docs/setup.md) first, then run cells from top to bottom. No quantum
hardware or remote account is required.


## Background

A fixed XOR mask flips selected measurement bits. Applying the same key again
removes those flips. The key remains fixed over the shots of one job.

Dynamic circuits can measure qubits before execution ends, use classical results
in conditions and reuse physical wires. Some of this behavior can be legitimate, but
some may expose information unintentionally. You must investigate which records
control the injected behavior and what those operations do.

Counts are frequencies: an entry with value 500 represents 500 observations.
Preserve leading zeros and bit order.


## 1. Load the necessary files

The index pairs circuit files with their supplied counts and lists the case settings.


In [ ]:
from pathlib import Path
import sys
from importlib.metadata import version

candidates = [Path.cwd(), Path.cwd() / 'quantum_trojan_challenge_multi', *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'cases.json').is_file()
             and (p / 'TROJAN_CHALLENGE.ipynb').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Start from the supplied challenge folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from challenge_utils import load_cases, load_counts, project_register_counts, validate_answer

CASES = load_cases(ROOT)
print('Python:', sys.version.split()[0])
print('Qiskit:', version('qiskit'), '| Aer:', version('qiskit-aer'))
for case_id, case in CASES.items():
    print(case_id, '| readout bits:', case['key_bits'], '| supplied shots:', case['shots'])


## 2. Select a case and load its evidence

Read the [full brief](challenges/01_silent_witness/README.md). Repeat your
investigation for all six cases, or develop code that handles them together.
You may inspect, modify, simplify and simulate copies of the supplied circuits.


In [ ]:
from qiskit import qasm3

CASE_ID = 'circuit1'  # circuit1 ... circuit6
case = CASES[CASE_ID]
circuit = qasm3.loads((ROOT / case['circuit']).read_text())
counts = load_counts(ROOT / case['measurements'])
print('Case:', CASE_ID)
print('Active quantum wires:', circuit.num_qubits)
print('Full classical record width:', circuit.num_clbits)
print('Supplied readout register:', case['readout_register'])
print('Supplied outcome width:', len(next(iter(counts))))
print('Distinct outcomes:', len(counts), '| total shots:', sum(counts.values()))
print('Example (word, frequency):', next(iter(counts.items())))


## 3. Inspect the circuit and counts

The supplied measurement words and submitted XOR key refer to the final values
of readout register `c`.

For `c`, displayed words run from the highest-index bit down to `c[0]`.
The submitted key must follow that same displayed order. Trigger descriptions
must identify registers and zero-based QASM indices, rather than ambiguous
character positions.

Circuit drawings and histograms can be large. Enable the examples below only
when useful; a plot of the most common words is not the complete distribution.


In [ ]:
DRAW_CIRCUIT = False
PLOT_COUNTS = False
if DRAW_CIRCUIT:
    print(circuit.draw(output='text', fold=120))
if PLOT_COUNTS:
    from qiskit.visualization import plot_histogram
    most_common = dict(sorted(counts.items(), key=lambda item: item[1], reverse=True)[:20])
    display(plot_histogram(most_common, title='Twenty most frequent supplied words'))


## 4. Optional local simulation

Use the supplied counts as the shared reference. You do not need to regenerate
all 8,192 shots. Large dynamic circuits can take substantial time and memory;
start with a small local batch.

The fixed seed below makes your own experiment repeatable. Exact counts can
differ between sampling algorithms and backend versions even with the same
seed; compare statistical relationships rather than dictionary equality.

Local simulator results and the supplied measurement words both use the final
values of readout register `c`.


In [ ]:
RUN_SIMULATION = False
SIMULATION_SHOTS = 64
SIMULATION_SEED = case['simulation_seed']  # 2026
local_counts = None
if RUN_SIMULATION:
    from qiskit import transpile
    from qiskit_aer import AerSimulator
    simulator = AerSimulator(method='statevector', max_parallel_threads=4,
                             max_parallel_shots=1, max_memory_mb=2048)
    runnable = transpile(circuit, simulator, seed_transpiler=2026)
    full_counts = simulator.run(runnable, shots=SIMULATION_SHOTS,
                                seed_simulator=SIMULATION_SEED).result().get_counts()
    local_counts = project_register_counts(circuit, full_counts, case['readout_register'])
    print('Local shots:', sum(local_counts.values()))
    print('Example projected records:', list(local_counts.items())[:5])
else:
    print('Using supplied counts; no new simulation requested.')


## 5. Your investigation

Develop your recovery method here. The same key applies to all supplied shots
of one case, but it need not apply to another case. Distinguish measurements
that carry data, execution records, and conditions that activate injected
operations. Support your conclusions using the circuit, counts, or experiments.

The starter notebook does not identify the trigger or provide a recovery algorithm.


In [ ]:
# YOUR ANALYSIS: work with circuit and counts, or iterate through CASES.
# Keep the original evidence files unchanged.
# Record your key and trigger findings in answers below.


## 6. Prepare your answers

For each case, provide the complete key and a trigger description. Identify
classical bits using register names and zero-based QASM indices. Explain both
the activation condition and what the injected operations do. If a recorded
indicator is derived from other execution records, explain that relationship.

These fields are intentionally blank. Decoded outcomes or count distributions
are not required. The helper below checks formatting only.


In [ ]:
answers = {
    case_id: {
        'key': '',
        'trigger': {
            'bits': [],  # entries have fields 'register' and 'index'
            'activation_condition': '',
            'explanation': '',
        },
    }
    for case_id in CASES
}
# YOUR ANSWERS: fill answers[CASE_ID], or load findings from your recovery code.
# For example:
# answers['circuit1'] = {
#     'key': '001101',  # Dummy key
#     'trigger': {
#         'bits': [
#             {'register': 'r2', 'index': 3},
#         ],
#         'activation_condition': 'r2[3] == 1',
#         'explanation': (
#             'Short explanation'
#             'Second sentence'
#         ),
#     },
# }


In [ ]:
for case_id, answer in answers.items():
    if not answer['key'] and not answer['trigger']['bits']:
        print(case_id + ': no answer entered.')
        continue
    try:
        validate_answer(answer, CASES[case_id]['key_bits'])
        print(case_id + ': complete format; correctness not checked.')
    except ValueError as error:
        print(case_id + ': incomplete draft — ' + str(error))


## 7. Save a local submission draft

Submit your findings, the code used to recover them, and a write-up covering all
six cases. A shared solution script and combined write-up are acceptable.
Templates are in `templates/`. Partial work should be identified in the
write-up. The draft can retain incomplete entries.

This cell writes a local file only. It neither submits nor grades anything.


In [ ]:
WRITE_DRAFT = False
if WRITE_DRAFT:
    import json
    destination = ROOT / 'submissions/submission.json'
    destination.parent.mkdir(exist_ok=True)
    payload = {'circuits': answers, 'code': 'solution.py', 'writeup': 'writeup.md'}
    destination.write_text(json.dumps(payload, indent=2) + '\n')
    print('Saved:', destination.relative_to(ROOT))
else:
    print('No submission file written.')


## Submit

Follow the event organizer's submission instructions. Full credit requires all
six keys, justified trigger identification and explanations, recovery code and a
write-up. Partial credit may be awarded for a reasonable method supported by
clear explanations and working code, even if not all keys are fully recovered.

This project contains no answer key or online grading endpoint.
